# Named Entity Recognition — Azure AI Language

This notebook uses **Azure AI Language** to identify and categorize *named entities* in text — people, organizations, locations, dates, quantities, and more.

NER is useful for information extraction, knowledge graph population, and building search indexes.

In [1]:
%pip install azure-ai-textanalytics azure-core python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\w\repos\foundry-tools\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
api_key = os.environ["AZURE_LANGUAGE_KEY"]

client = TextAnalyticsClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
print("Language client ready.")

Language client ready.


In [3]:
# Sample documents for NER
documents = [
    "Satya Nadella is the CEO of Microsoft, headquartered in Redmond, Washington.",
    "The Eiffel Tower was built in 1889 and stands 330 meters tall in Paris, France.",
    "On January 15, 2024, OpenAI announced a new partnership worth $10 billion with Microsoft Azure.",
]

print(f"Recognizing entities in {len(documents)} documents...")

Recognizing entities in 3 documents...


In [4]:
# Perform Named Entity Recognition
results = client.recognize_entities(documents=documents)

for i, result in enumerate(results):
    if result.is_error:
        print(f"Document {i + 1} error: {result.error.code} - {result.error.message}")
        continue

    print(f"\n--- Document {i + 1} ---")
    print(f"Text: {documents[i]}")
    print("Entities:")
    for entity in result.entities:
        print(f"  '{entity.text}' → category: {entity.category}, "
              f"subcategory: {entity.subcategory or '—'}, "
              f"confidence: {entity.confidence_score:.4f}")


--- Document 1 ---
Text: Satya Nadella is the CEO of Microsoft, headquartered in Redmond, Washington.
Entities:
  'Satya Nadella' → category: Person, subcategory: —, confidence: 1.0000
  'CEO' → category: PersonType, subcategory: —, confidence: 0.9000
  'Microsoft' → category: Organization, subcategory: —, confidence: 0.9900
  'Redmond' → category: Location, subcategory: City, confidence: 0.9600
  'Washington' → category: Location, subcategory: State, confidence: 0.5200

--- Document 2 ---
Text: The Eiffel Tower was built in 1889 and stands 330 meters tall in Paris, France.
Entities:
  'Eiffel Tower' → category: Location, subcategory: —, confidence: 0.9900
  '1889' → category: DateTime, subcategory: DateRange, confidence: 1.0000
  '330 meters' → category: Quantity, subcategory: Dimension, confidence: 1.0000
  'Paris' → category: Location, subcategory: City, confidence: 0.7400
  'France' → category: Location, subcategory: CountryRegion, confidence: 0.9100

--- Document 3 ---
Text: On J

In [5]:
# Group entities by category across all documents
from collections import defaultdict

by_category = defaultdict(list)

results2 = client.recognize_entities(documents=documents)
for result in results2:
    if not result.is_error:
        for entity in result.entities:
            by_category[entity.category].append(entity.text)

print("\nEntities grouped by category:")
for category, entities in sorted(by_category.items()):
    unique_entities = sorted(set(entities))
    print(f"  {category}: {', '.join(unique_entities)}")


Entities grouped by category:
  DateTime: 1889, January 15, 2024
  Event: OpenAI
  Location: Eiffel Tower, France, Paris, Redmond, Washington
  Organization: Microsoft
  Person: Satya Nadella
  PersonType: CEO
  Product: Microsoft Azure
  Quantity: $10 billion, 330 meters
